# DeepGuard v2 · Phase 1.1 — FaceForensics++ data audit

**What this notebook does:** measures every video in the attached FaceForensics++ copy and checks it against the official dataset facts, *before* any preprocessing or training.

| Cell | Check |
|---|---|
| 1 | Setup and tool check |
| 2 | Folder tree: what is in the dataset (videos or frames, sizes, layout) |
| 3 | Inventory: every file parsed into method / compression / IDs |
| 4 | Completeness against the official split files (1,000 originals, 1,000 fakes per method) + official-split leakage |
| 5 | ffprobe on every video + SHA-256 checksum (integrity, manifest) |
| 6 | Metadata tables (resolution, fps, frames, bitrate, audio) |
| 7 | Pair consistency: does each fake match its target video? (frame count, fps → the "weird speed" question) |
| 8 | Metadata shortcut audit: can file properties alone tell real from fake? |
| 9 | Face detector set-up and self-test |
| 10 | Visual check of every c23 fake against its original: where the pixels changed, how many faces, which face was manipulated |
| 11 | Visual summaries |
| 12 | Contact sheets of flagged videos (to look at with your own eyes) |
| 13 | Report + zip for download |

**Settings:** Accelerator **None**, Internet **On**. Expected run time on CPU: about 1–2 hours. Use **Save Version → Save & Run All** so it runs in the background.
Every slow cell caches its results in `/kaggle/working/ffpp_audit`, so a re-run continues where it stopped.

In [ ]:
# ===== CELL 1: Setup — imports, settings, tool check =====
# Loads libraries, defines where data is read from and where results are written,
# and checks that ffprobe (video metadata tool) is available.
import os, re, sys, json, time, hashlib, subprocess, shutil, random, zipfile, urllib.request
import multiprocessing as mp
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
import numpy as np
import pandas as pd
import cv2

T0 = time.time()
def log(msg):
    """Print a message with elapsed time, so you can see progress while a cell runs."""
    m, s = divmod(int(time.time() - T0), 60)
    print(f"[{m:3d}m{s:02d}s] {msg}", flush=True)

# ---------------- settings you may change ----------------
INPUT_ROOT = Path(os.environ.get("DG_INPUT_ROOT", "/kaggle/input"))   # Kaggle mounts attached datasets here
OUT = Path(os.environ.get("DG_OUT", "/kaggle/working/ffpp_audit"))      # every result is written here
VISUAL_COMPRESSION = "c23"          # visual checks use c23 (c40 is the same content, more compressed)
VISUAL_MAX_FAKES_PER_METHOD = None  # None = every fake; set e.g. 100 for a quick trial run
DFD_VISUAL_MAX = 600                # DeepFakeDetection fakes are 1080p and slow to decode → sample this many
FRAME_IDX = [5, 45, 85, 125]        # frame numbers compared between a fake and its original (early → fast)
DIFF_THRESHOLD = 25                 # grey-level change that counts as "manipulated pixel" (after blur)
N_WORKERS = max(1, os.cpu_count() or 1)
SEED = 2026
# ----------------------------------------------------------

OFFICIAL_BASE = "https://raw.githubusercontent.com/ondyari/FaceForensics/master/dataset/"
OFFICIAL_LOCAL = os.environ.get("DG_OFFICIAL_DIR")  # only used for offline testing
YUNET_URLS = [
    "https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx",
    "https://media.githubusercontent.com/media/opencv/opencv_zoo/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx",
]
FF_METHODS = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures", "FaceShifter"]
VIDEO_EXT = {".mp4", ".avi", ".mov", ".mkv", ".webm"}
IMAGE_EXT = {".png", ".jpg", ".jpeg", ".bmp"}

OUT.mkdir(parents=True, exist_ok=True)
(OUT / "figures").mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
FFPROBE = shutil.which("ffprobe")
MP_CTX = mp.get_context("fork")     # worker processes inherit the notebook's functions

log(f"Python {sys.version.split()[0]} | OpenCV {cv2.__version__} | pandas {pd.__version__} | CPU workers: {N_WORKERS}")
log(f"Input root: {INPUT_ROOT}  (exists: {INPUT_ROOT.exists()})")
log(f"Output dir: {OUT}")
log(f"ffprobe: {FFPROBE or 'NOT FOUND'}")
assert FFPROBE, "ffprobe not found. On Kaggle it is pre-installed; if missing run: !apt-get -y install ffmpeg"
assert INPUT_ROOT.exists(), "Input folder not found — attach the FF++ dataset with 'Add Input'."

In [ ]:
# ===== CELL 2: Folder tree — what is actually inside the dataset? =====
# Walks every attached folder, counts files by type and size, and prints a compact tree.
# This answers: are these videos or extracted frames? which folders hold what?
rows = []
for dirpath, dirnames, filenames in os.walk(INPUT_ROOT):
    dirnames.sort()
    for fn in filenames:
        p = Path(dirpath) / fn
        try:
            size = p.stat().st_size
        except OSError:
            size = -1
        rows.append((str(p), p.suffix.lower(), size))
files = pd.DataFrame(rows, columns=["path", "ext", "bytes"])
log(f"Total files: {len(files):,} | total size: {files.bytes.clip(lower=0).sum()/1e9:.2f} GB")
print("\nFiles by extension:")
print(files.groupby("ext").agg(files=("path", "size"), GB=("bytes", lambda b: round(b.sum()/1e9, 2)))
      .sort_values("files", ascending=False).to_string())

# Compact tree: one line per folder that directly contains files, with counts by type
files["folder"] = files.path.map(lambda s: str(Path(s).parent.relative_to(INPUT_ROOT)))
tree = (files.groupby(["folder", "ext"]).size().unstack(fill_value=0))
tree["GB"] = files.groupby("folder").bytes.sum() / 1e9
tree = tree.sort_index()
n_vid = files.ext.isin(VIDEO_EXT).sum(); n_img = files.ext.isin(IMAGE_EXT).sum()
print(f"\nVideos: {n_vid:,} | images: {n_img:,} | other: {len(files)-n_vid-n_img:,}")
print("→ The dataset is", "VIDEOS" if n_vid > n_img else "IMAGES/FRAMES" if n_img else "UNKNOWN",
      "(images are expected only if someone extracted frames)")
# Folders holding many single-video image folders are summarised by their parent to keep the print short
leaf_img = tree[(tree.drop(columns="GB").sum(axis=1) > 0)]
print(f"\nFolders that contain files: {len(leaf_img):,}. First 120:")
with pd.option_context("display.width", 250, "display.max_columns", 30):
    print(tree.head(120).round(3).to_string())
tree.to_csv(OUT / "tree.csv")
files.to_csv(OUT / "all_files.csv", index=False)
log("Saved tree.csv and all_files.csv")

In [ ]:
# ===== CELL 3: Inventory — parse every video into dataset / method / compression / IDs =====
# Official naming (FF++ dataset README):
#   originals (YouTube)  : 000.mp4 … 999.mp4
#   FF++ fakes           : <target>_<source>.mp4   e.g. 071_054.mp4 = video 071 with input from video 054
#   DFD originals (actors): <actor>__<scene>.mp4
#   DFD fakes            : <target actor>_<source actor>__<scene>__<8-char experiment id>.mp4
RE_FF_ORIG = re.compile(r"^(\d{3})$")
RE_FF_FAKE = re.compile(r"^(\d{3})_(\d{3})$")
RE_DFD_ORIG = re.compile(r"^(\d{2})__(.+)$")
RE_DFD_FAKE = re.compile(r"^(\d{2})_(\d{2})__(.+)__([A-Za-z0-9]{8})$")
RE_COMP = re.compile(r"(?:^|[^a-z0-9])(c23|c40|c0|raw|hq|lq)(?:$|[^a-z0-9])")

def method_from_path(low):
    """Find the manipulation method name inside the (lower-case) path."""
    if "deepfakedetection" in low or "deep_fake_detection" in low:
        return "DeepFakeDetection"
    for m in FF_METHODS:
        if m.lower() in low:
            return m
    return None

def comp_from_path(low):
    """Find the compression level (c23 / c40 / raw) inside the path."""
    hits = RE_COMP.findall(low)
    if not hits:
        return "unknown"
    h = hits[-1]
    return {"hq": "c23", "lq": "c40", "c0": "raw"}.get(h, h)

def parse_item(path, stem, kind):
    """Turn one video file (or one folder of frames) into a labelled inventory row."""
    rel = str(Path(path).relative_to(INPUT_ROOT))
    low = rel.lower()
    r = dict(path=path, rel=rel, kind=kind, stem=stem, compression=comp_from_path(low),
             dataset=None, label=None, method=None, vid=None, target=None, source=None,
             scene=None, exp_id=None, note="")
    pm = method_from_path(low)
    if m := RE_DFD_FAKE.match(stem):
        r.update(dataset="DFD", label="fake", method="DeepFakeDetection",
                 target=m[1], source=m[2], scene=m[3], exp_id=m[4], vid=stem)
    elif m := RE_DFD_ORIG.match(stem):
        r.update(dataset="DFD", label="real", method="original_actors", target=m[1], scene=m[2], vid=stem)
    elif m := RE_FF_FAKE.match(stem):
        r.update(dataset="FFPP", label="fake", method=pm or "UNKNOWN", target=m[1], source=m[2], vid=stem)
        if pm in (None, "DeepFakeDetection"):
            r["note"] = "fake name but method not found in path"
    elif m := RE_FF_ORIG.match(stem):
        r.update(dataset="FFPP", label="real", method="original", target=m[1], vid=stem)
        if pm is not None:
            r["note"] = f"original-style name inside a '{pm}' folder"
    else:
        r.update(dataset="UNPARSED", note="name does not match any official pattern")
    return r

items = []
for p in files.loc[files.ext.isin(VIDEO_EXT), "path"]:
    items.append(parse_item(p, Path(p).stem, "video"))
# Frames stored as images: one folder per video → treat the folder as the item
img = files[files.ext.isin(IMAGE_EXT)]
for folder, g in img.groupby(img.path.map(lambda s: str(Path(s).parent))):
    r = parse_item(folder, Path(folder).name, "frames_folder")
    r["n_images"] = len(g)
    items.append(r)
inv = pd.DataFrame(items)
log(f"Inventory rows: {len(inv):,} (videos: {(inv.kind=='video').sum():,}, frame folders: {(inv.kind=='frames_folder').sum():,})")

print("\nCount by dataset / method / compression / kind:")
print(inv.pivot_table(index=["dataset", "method"], columns=["compression", "kind"], values="path",
                      aggfunc="count", fill_value=0).to_string())
bad = inv[(inv.dataset == "UNPARSED") | (inv.note != "")]
print(f"\nRows needing attention: {len(bad)}")
if len(bad):
    print(bad[["rel", "note"]].head(40).to_string(index=False))
dups = inv[inv.kind == "video"].groupby(["dataset", "method", "compression", "stem"]).size()
dups = dups[dups > 1]
print(f"Same video name appearing twice in one method/compression: {len(dups)}")
if len(dups): print(dups.head(20).to_string())
inv.to_csv(OUT / "inventory.csv", index=False)
log("Saved inventory.csv")

In [ ]:
# ===== CELL 4: Completeness vs the official files + official-split leakage =====
# Downloads the official split files (train/val/test pairs) and conversion_dict.json
# (which YouTube video each sequence was cut from) from the FaceForensics GitHub.
def get_official(name):
    """Read an official FF++ metadata file (from GitHub, or a local folder when testing offline)."""
    if OFFICIAL_LOCAL:
        return json.load(open(Path(OFFICIAL_LOCAL) / name.replace("/", "_")))
    with urllib.request.urlopen(OFFICIAL_BASE + name, timeout=60) as f:
        return json.loads(f.read().decode())

splits = {s: get_official(f"splits/{s}.json") for s in ["train", "val", "test"]}
conv = get_official("conversion_dict.json")
id2split = {i: s for s, pairs in splits.items() for p in pairs for i in p}
pair_of = {}                       # each ID → its partner in the official pair list
for s, pairs in splits.items():
    for a, b in pairs:
        pair_of[a], pair_of[b] = b, a
log("Official splits: " + ", ".join(f"{s}={len(p)} pairs/{2*len(p)} videos" for s, p in splits.items()))

exp_orig = {f"{i:03d}" for i in range(1000)}
exp_fake = {f"{a}_{b}" for p in splits.values() for a, b in p} | {f"{b}_{a}" for p in splits.values() for a, b in p}
print(f"Expected: {len(exp_orig)} originals, {len(exp_fake)} fakes per method (each pair in both directions)\n")

main_kind = "video" if (inv.kind == "video").sum() >= (inv.kind == "frames_folder").sum() else "frames_folder"
vids = inv[(inv.dataset == "FFPP") & (inv.kind == main_kind)]   # count videos (or frame folders if that is all there is)
comp_rows = []
for (method, comp), g in vids.groupby(["method", "compression"]):
    found = set(g.stem)
    exp = exp_orig if method == "original" else exp_fake
    comp_rows.append(dict(method=method, compression=comp, found=len(found), expected=len(exp),
                          missing=len(exp - found), extra=len(found - exp),
                          missing_examples=",".join(sorted(exp - found)[:8]),
                          extra_examples=",".join(sorted(found - exp)[:8])))
completeness = pd.DataFrame(comp_rows)
print(completeness.to_string(index=False))

# c23 vs c40: the same file names should exist in both
print("\nc23 vs c40 name parity (should be identical sets):")
for method, g in vids.groupby("method"):
    a = set(g[g.compression == "c23"].stem); b = set(g[g.compression == "c40"].stem)
    if a or b:
        print(f"  {method:16s} c23={len(a):5d} c40={len(b):5d} only-c23={len(a-b):4d} only-c40={len(b-a):4d}")

# DeepFakeDetection (no official split; expected ≈363 originals of 28 actors in 16 scenes, 3,068 fakes)
dfd = inv[inv.dataset == "DFD"]
if len(dfd):
    for comp, g in dfd.groupby("compression"):
        r, f = g[g.label == "real"], g[g.label == "fake"]
        no_orig = (f.target + "__" + f.scene).isin(set(r.stem))
        print(f"\nDFD {comp}: originals={len(r)} (actors={r.target.nunique()}, scenes={r.scene.nunique()}) | "
              f"fakes={len(f)} | fakes whose target original is present: {no_orig.sum()}/{len(f)}")
else:
    print("\nNo DeepFakeDetection files found in this copy.")

# Official-split leakage: sequences cut from the SAME YouTube video placed in different splits
yt = defaultdict(list)
for k, v in conv.items():
    yt[v.rsplit(" ", 1)[0]].append(k)
multi = {k: sorted(v) for k, v in yt.items() if len(v) > 1}
leak = [(k, [(i, id2split[i]) for i in v]) for k, v in multi.items() if len({id2split[i] for i in v}) > 1]
print(f"\n1,000 sequences come from {len(yt)} YouTube videos; {len(multi)} videos gave >1 sequence.")
print(f"YouTube videos whose sequences sit in DIFFERENT official splits (same person+scene across splits): {len(leak)}")
for k, v in leak:
    print("   ", k, v)
completeness.to_csv(OUT / "completeness.csv", index=False)
json.dump({"leak_youtube_cross_split": leak}, open(OUT / "official_split_leakage.json", "w"), indent=1)
log("Saved completeness.csv and official_split_leakage.json")

In [ ]:
# ===== CELL 5: ffprobe + SHA-256 for every video (parallel, resumable) =====
# For each video: codec, resolution, frame rate, frame count (counted, not just the header),
# duration, bitrate, number of audio streams, encoder tag, file size and SHA-256 checksum.
# Results are cached in probe.csv; re-running skips videos already done.
def fps_val(s):
    """'30000/1001' → 29.97"""
    try:
        a, b = s.split("/"); return float(a) / float(b) if float(b) else np.nan
    except Exception:
        return np.nan

def probe_one(path):
    """Measure one video file. Never raises: errors are recorded in the 'error' column."""
    r = dict(path=path, error="")
    try:
        h = hashlib.sha256()
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(8 << 20), b""):
                h.update(chunk)
        r["sha256"] = h.hexdigest()
        r["bytes"] = os.path.getsize(path)
        cmd = [FFPROBE, "-v", "error", "-count_packets", "-show_entries",
               "stream=codec_type,codec_name,width,height,pix_fmt,r_frame_rate,avg_frame_rate,nb_frames,nb_read_packets,bit_rate,duration"
               ":format=duration,bit_rate,format_name:format_tags=encoder", "-of", "json", path]
        j = json.loads(subprocess.run(cmd, capture_output=True, text=True, timeout=300).stdout or "{}")
        st = j.get("streams", []); fm = j.get("format", {})
        v = next((s for s in st if s.get("codec_type") == "video"), None)
        r["n_audio"] = sum(s.get("codec_type") == "audio" for s in st)
        r["container_duration"] = float(fm.get("duration", "nan"))
        r["container_bitrate"] = float(fm.get("bit_rate", "nan"))
        r["encoder"] = (fm.get("tags") or {}).get("encoder", "")
        if v is None:
            r["error"] = "no video stream"
        else:
            r.update(codec=v.get("codec_name"), width=int(v.get("width", 0)), height=int(v.get("height", 0)),
                     pix_fmt=v.get("pix_fmt"), r_fps=fps_val(v.get("r_frame_rate", "")),
                     avg_fps=fps_val(v.get("avg_frame_rate", "")),
                     nb_frames_header=float(v.get("nb_frames", "nan")),
                     n_frames=float(v.get("nb_read_packets", "nan")),
                     v_bitrate=float(v.get("bit_rate", "nan")), v_duration=float(v.get("duration", "nan")))
            if not r["n_frames"] or np.isnan(r["n_frames"]):
                r["error"] = "zero frames"
    except Exception as e:
        r["error"] = f"{type(e).__name__}: {e}"[:200]
    return r

probe_csv = OUT / "probe.csv"
todo = inv.loc[inv.kind == "video", "path"].tolist()
done = pd.read_csv(probe_csv) if probe_csv.exists() else pd.DataFrame(columns=["path"])
todo = [p for p in todo if p not in set(done.path)]
log(f"Videos to probe: {len(todo):,} (already cached: {len(done):,})")
buf, t1 = [], time.time()
with ProcessPoolExecutor(N_WORKERS, mp_context=MP_CTX) as ex:
    futs = [ex.submit(probe_one, p) for p in todo]
    for k, fu in enumerate(as_completed(futs), 1):
        buf.append(fu.result())
        if k % 500 == 0 or k == len(futs):
            done = pd.concat([done, pd.DataFrame(buf)], ignore_index=True); buf = []
            done.to_csv(probe_csv, index=False)
            rate = k / max(time.time() - t1, 1e-6)
            log(f"  probed {k:,}/{len(futs):,}  ({rate:.1f}/s, ETA {(len(futs)-k)/max(rate,1e-6)/60:.1f} min)")
probe = pd.read_csv(probe_csv)
meta = inv.merge(probe, on="path", how="left", suffixes=("", "_p"))
errs = meta[(meta.kind == "video") & (meta.error.fillna("") != "")]
log(f"Probe done. Videos with errors (unreadable / no frames): {len(errs)}")
if len(errs): print(errs[["rel", "error"]].head(30).to_string(index=False))
dup_hash = meta[meta.kind == "video"].groupby("sha256").rel.apply(list)
dup_hash = dup_hash[dup_hash.map(len) > 1]
print(f"Byte-identical duplicate files (same SHA-256): {len(dup_hash)} groups")
for h, lst in list(dup_hash.items())[:15]:
    print("   ", lst)
meta.to_csv(OUT / "meta.csv", index=False)
meta.loc[meta.kind == "video", ["rel", "bytes", "sha256"]].to_csv(OUT / "MANIFEST.csv", index=False)
log("Saved meta.csv and MANIFEST.csv (relative path, size, SHA-256 — our Layer-1 checksum manifest)")

In [ ]:
# ===== CELL 6: Metadata tables — what do the videos look like? =====
v = meta[(meta.kind == "video") & (meta.error.fillna("") == "")].copy()
v["res"] = v.width.astype("Int64").astype(str) + "x" + v.height.astype("Int64").astype(str)
v["fps_r"] = v.avg_fps.round(3)
v["vfr"] = (v.r_fps - v.avg_fps).abs() > 0.01       # variable frame rate if the two rates disagree
v["dur"] = v.n_frames / v.avg_fps
v["bpp"] = v.v_bitrate / (v.width * v.height * v.avg_fps)  # bits per pixel per frame
g = v.groupby(["dataset", "method", "compression"])
summary = g.agg(videos=("rel", "size"),
                frames_min=("n_frames", "min"), frames_med=("n_frames", "median"), frames_max=("n_frames", "max"),
                fps_values=("fps_r", "nunique"), fps_med=("fps_r", "median"),
                dur_med_s=("dur", "median"), res_values=("res", "nunique"),
                kbps_med=("v_bitrate", lambda x: round(x.median() / 1e3)), bpp_med=("bpp", "median"),
                audio_videos=("n_audio", lambda x: int((x > 0).sum())), vfr_videos=("vfr", "sum"),
                header_vs_counted_mismatch=("nb_frames_header", lambda x: 0))
summary["header_vs_counted_mismatch"] = g[["nb_frames_header", "n_frames"]].apply(
    lambda d: int(((d.nb_frames_header - d.n_frames).abs() > 0).sum()))
with pd.option_context("display.width", 250, "display.max_columns", 30):
    print(summary.round(4).to_string())
print("\nMost common frame rates:", v.fps_r.value_counts().head(12).to_dict())
print("Most common resolutions:", v.res.value_counts().head(12).to_dict())
print("Codecs:", v.codec.value_counts().to_dict(), "| pixel formats:", v.pix_fmt.value_counts().to_dict())
print("Encoder tags:", v.encoder.fillna("").value_counts().head(8).to_dict())
summary.to_csv(OUT / "metadata_summary.csv")
log("Saved metadata_summary.csv")

In [ ]:
# ===== CELL 7: Pair consistency — frame count and fps of each fake vs its target and source originals =====
# Pre-registered hypotheses (from the official compress.py):
#   H1: a fake is encoded with the SOURCE video's fps ("We take the fps of the source video"),
#       although its frames come from the TARGET video → it plays faster/slower than the real target.
#   H2: a fake has the same number of frames as its target (FaceSwap may be shorter: it stops when one video ends).
orig = v[(v.dataset == "FFPP") & (v.label == "real")].set_index(["compression", "target"])
orig = orig[~orig.index.duplicated()]     # if a name appears twice, use the first copy
fk = v[(v.dataset == "FFPP") & (v.label == "fake")].copy()
def lookup(comp, vid, col):
    try:
        return orig.loc[(comp, vid), col]
    except KeyError:
        return np.nan
for who, key in [("tgt", "target"), ("src", "source")]:
    fk[f"{who}_fps"] = [lookup(c, i, "avg_fps") for c, i in zip(fk.compression, fk[key])]
    fk[f"{who}_frames"] = [lookup(c, i, "n_frames") for c, i in zip(fk.compression, fk[key])]
close = lambda a, b: (a - b).abs() < 0.01
fk["fps_eq_tgt"] = close(fk.avg_fps, fk.tgt_fps); fk["fps_eq_src"] = close(fk.avg_fps, fk.src_fps)
fk["fps_case"] = np.select([fk.fps_eq_tgt & fk.fps_eq_src, fk.fps_eq_src, fk.fps_eq_tgt, fk.tgt_fps.isna()],
                           ["tgt=src (same)", "= SOURCE only", "= TARGET only", "no original"], "neither")
fk["frames_case"] = np.select([fk.n_frames == fk.tgt_frames,
                               fk.n_frames == np.minimum(fk.tgt_frames, fk.src_frames),
                               fk.n_frames < fk.tgt_frames, fk.tgt_frames.isna()],
                              ["= target", "= min(target,source)", "shorter (other)", "no original"], "longer/other")
fk["speed_vs_target"] = fk.avg_fps / fk.tgt_fps        # >1 → the fake plays faster than the real target video
print("fps of each fake compared with its originals (H1):")
print(pd.crosstab([fk.method, fk.compression], fk.fps_case).to_string())
print("\nFrame count of each fake compared with its originals (H2):")
print(pd.crosstab([fk.method, fk.compression], fk.frames_case).to_string())
sp = fk.speed_vs_target
print(f"\nPlayback speed of fake vs its real target: {(sp.sub(1).abs() > 0.01).mean():.1%} of fakes differ; "
      f"range {sp.min():.3f}× … {sp.max():.3f}×")
print(fk.loc[sp.sub(1).abs().sort_values(ascending=False).index[:15],
             ["rel", "avg_fps", "tgt_fps", "src_fps", "speed_vs_target", "n_frames", "tgt_frames"]].round(3).to_string(index=False))
fk.to_csv(OUT / "pair_consistency.csv", index=False)
log("Saved pair_consistency.csv")

In [ ]:
# ===== CELL 8: Metadata shortcut audit — can file properties alone separate real from fake? =====
# A gradient-boosting model sees ONLY metadata (no pixels). If it reaches AUC > 0.60, a detector could
# "cheat" with those properties, so Phase 1 must neutralise them. Folds keep each official pair together.
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
FEATS = ["width", "height", "avg_fps", "n_frames", "dur", "v_bitrate", "bpp", "bytes"]
def shortcut_auc(d):
    """Out-of-fold AUC of a metadata-only classifier, plus AUC with each feature removed."""
    X, y = d[FEATS].astype(float).values, (d.label == "fake").astype(int).values
    grp = d.target.map(lambda t: "-".join(sorted([t, pair_of.get(t, t)])))
    k = min(5, grp.nunique())
    if k < 2:
        return np.nan, {}
    def oof(cols):
        p = np.zeros(len(d))
        for tr, te in GroupKFold(k).split(X, y, grp):
            m = HistGradientBoostingClassifier(max_iter=200, random_state=SEED).fit(X[tr][:, cols], y[tr])
            p[te] = m.predict_proba(X[te][:, cols])[:, 1]
        return roc_auc_score(y, p)
    full = oof(list(range(len(FEATS))))
    drops = {f: full - oof([j for j in range(len(FEATS)) if j != i]) for i, f in enumerate(FEATS)}
    return full, drops
res = []
ff = v[v.dataset == "FFPP"]
for comp in sorted(ff.compression.unique()):
    for method in ["ALL"] + [m for m in FF_METHODS if m in set(ff.method)]:
        d = ff[(ff.compression == comp) & ((ff.label == "real") | (ff.method == method) | (method == "ALL"))]
        if d.label.nunique() < 2:
            continue
        try:
            auc, drops = shortcut_auc(d)
        except ValueError as e:          # e.g. a fold with only one class (tiny trial runs)
            log(f"  {comp} {method}: skipped ({e})"); continue
        top = sorted(drops.items(), key=lambda kv: -kv[1])[:3]
        res.append(dict(compression=comp, fakes=method, n=len(d), auc=round(auc, 3),
                        verdict="FLAG > 0.60" if auc > 0.60 else "ok" if auc == auc else "n/a",
                        top_features=", ".join(f"{k}(+{d_:.3f})" for k, d_ in top)))
        log(f"  {comp} real vs {method:15s} metadata-only AUC = {auc:.3f}")
shortcut = pd.DataFrame(res, columns=["compression", "fakes", "n", "auc", "verdict", "top_features"])
print(shortcut.to_string(index=False))
shortcut.to_csv(OUT / "shortcut_audit.csv", index=False)
log("Saved shortcut_audit.csv")

In [ ]:
# ===== CELL 9: Face detector set-up (OpenCV YuNet) + self-test =====
# YuNet finds every face in a frame (including small or side faces), so we can count faces per video.
YUNET_PATH = Path("/kaggle/working/yunet.onnx") if Path("/kaggle/working").exists() else OUT / "yunet.onnx"
STUB = os.environ.get("DG_STUB_FACES") == "1"     # offline testing only
if not STUB and not YUNET_PATH.exists():
    for u in YUNET_URLS:
        try:
            urllib.request.urlretrieve(u, YUNET_PATH)
            if YUNET_PATH.stat().st_size > 100_000:
                break
        except Exception as e:
            log(f"  download failed from {u[:60]}…: {e}")
FACES_OK = STUB or (YUNET_PATH.exists() and YUNET_PATH.stat().st_size > 100_000)
log(f"Face detector ready: {FACES_OK} ({'stub' if STUB else YUNET_PATH})")

_DET = {}
def detect_faces(img):
    """Return face boxes [x, y, w, h, score] in a BGR frame."""
    if STUB:   # test mode: 'faces' are magenta rectangles drawn by the test generator
        m = cv2.inRange(img, (180, 0, 180), (255, 90, 255))
        n, _, stats, _ = cv2.connectedComponentsWithStats(m)
        return [[*stats[i, :4], 0.99] for i in range(1, n) if stats[i, 4] > 50]
    h, w = img.shape[:2]
    d = _DET.get("d")
    if d is None:
        d = _DET["d"] = cv2.FaceDetectorYN.create(str(YUNET_PATH), "", (w, h), 0.7, 0.3, 50)
    d.setInputSize((w, h))
    _, faces = d.detect(img)
    if faces is None:
        return []
    return [[*f[:4].tolist(), float(f[-1])] for f in faces if f[2] >= 20 and f[3] >= 20]

if FACES_OK:
    one = v[(v.dataset == "FFPP") & (v.label == "real") & (v.compression == VISUAL_COMPRESSION)].head(1)
    if len(one):
        cap = cv2.VideoCapture(one.path.iloc[0]); ok, fr = cap.read(); cap.release()
        boxes = detect_faces(fr)
        log(f"Self-test on {one.rel.iloc[0]}: {len(boxes)} face(s) found → {np.round(boxes, 1).tolist()}")

In [ ]:
# ===== CELL 10: Visual check of every c23 fake against its original (parallel, resumable) =====
# For each fake and the SAME frame numbers of its target original:
#   diff  = |fake − original| (blurred)  → where pixels were changed (a home-made manipulation mask)
#   faces = every face in the fake frame → how many people, and which face overlaps the changed pixels
#   bg_diff = median difference OUTSIDE the changed area → ≈0 if the two videos are frame-aligned
def read_frames(path, idxs):
    """Decode a video sequentially and return the requested frame numbers (sequential = exact frames)."""
    want, out = set(idxs), {}
    cap = cv2.VideoCapture(path); i = 0
    while i <= max(idxs):
        ok, fr = cap.read()
        if not ok:
            break
        if i in want:
            out[i] = fr
        i += 1
    cap.release()
    return out

def overlap_share(mask, box, pad=0.15):
    """Share of all changed pixels that fall inside a (slightly enlarged) face box."""
    x, y, w, h = box[:4]; H, W = mask.shape
    x0, y0 = int(max(0, x - pad * w)), int(max(0, y - pad * h))
    x1, y1 = int(min(W, x + w * (1 + pad))), int(min(H, y + h * (1 + pad)))
    tot = mask.sum()
    return float(mask[y0:y1, x0:x1].sum() / tot) if tot else 0.0

def analyse_pair(job):
    """Compare one fake with its original at a few frames. Returns one row of measurements."""
    r = dict(path=job["path"], method=job["method"], error="")
    try:
        n = int(min(job["n_fake"], job["n_orig"]))
        idxs = [i for i in FRAME_IDX if i < n] or [0]
        F, O = read_frames(job["path"], idxs), read_frames(job["orig_path"], idxs)
        per = []
        for i in idxs:
            if i not in F or i not in O:
                continue
            f, o = F[i], O[i]
            if f.shape != o.shape:
                r["error"] = f"size mismatch {f.shape} vs {o.shape}"; break
            d = cv2.GaussianBlur(cv2.absdiff(f, o).max(axis=2), (5, 5), 0)
            m = (d > DIFF_THRESHOLD).astype(np.uint8)
            m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
            faces = detect_faces(f) if FACES_OK else []
            areas = [b[2] * b[3] for b in faces]
            shares = [overlap_share(m, b) for b in faces]
            k = int(np.argmax(shares)) if shares and max(shares) > 0.3 else -1
            rank = int(sorted(range(len(areas)), key=lambda j: -areas[j]).index(k)) if k >= 0 else -1
            per.append(dict(changed_frac=float(m.mean()), diff_p99=float(np.percentile(d, 99)),
                            bg_diff=float(np.median(d[m == 0])) if (m == 0).any() else np.nan,
                            n_faces=len(faces), manip_rank=rank,
                            manip_face_share=max(shares) if shares else 0.0,
                            changed_in_faces=min(1.0, sum(shares))))
        if per:
            P = pd.DataFrame(per)
            r.update(frames_checked=len(P), changed_frac=P.changed_frac.median(), diff_p99=P.diff_p99.median(),
                     bg_diff=P.bg_diff.median(), max_faces=int(P.n_faces.max()), min_faces=int(P.n_faces.min()),
                     multi_face_frames=float((P.n_faces >= 2).mean()),
                     manip_rank=int(P.manip_rank.mode().iloc[0]), manip_face_share=P.manip_face_share.median(),
                     changed_in_faces=P.changed_in_faces.median())
        elif not r["error"]:
            r["error"] = "no frames decoded"
    except Exception as e:
        r["error"] = f"{type(e).__name__}: {e}"[:200]
    return r

# Build the job list: FF++ fakes (all, or capped) + a sample of DFD fakes, each with its original
jobs = []
fv = v[(v.label == "fake") & (v.compression == VISUAL_COMPRESSION)]
ov = v[(v.label == "real") & (v.compression == VISUAL_COMPRESSION)].set_index(["dataset", "stem"])
for method, g in fv.groupby("method"):
    cap_n = DFD_VISUAL_MAX if method == "DeepFakeDetection" else VISUAL_MAX_FAKES_PER_METHOD
    if cap_n and len(g) > cap_n:
        g = g.sample(cap_n, random_state=SEED)
    for _, row in g.iterrows():
        key = (row.dataset, row.target if row.dataset == "FFPP" else f"{row.target}__{row.scene}")
        if key in ov.index:
            o = ov.loc[key]
            o = o.iloc[0] if isinstance(o, pd.DataFrame) else o
            jobs.append(dict(path=row.path, method=method, n_fake=row.n_frames, orig_path=o.path, n_orig=o.n_frames))
vis_csv = OUT / "visual.csv"
vdone = pd.read_csv(vis_csv) if vis_csv.exists() else pd.DataFrame(columns=["path"])
jobs = [j for j in jobs if j["path"] not in set(vdone.path)]
log(f"Fake/original pairs to check: {len(jobs):,} (cached: {len(vdone):,}) | faces: {FACES_OK}")
buf, t1 = [], time.time()
with ProcessPoolExecutor(N_WORKERS, mp_context=MP_CTX) as ex:
    futs = [ex.submit(analyse_pair, j) for j in jobs]
    for k, fu in enumerate(as_completed(futs), 1):
        buf.append(fu.result())
        if k % 250 == 0 or k == len(futs):
            vdone = pd.concat([vdone, pd.DataFrame(buf)], ignore_index=True); buf = []
            vdone.to_csv(vis_csv, index=False)
            rate = k / max(time.time() - t1, 1e-6)
            log(f"  checked {k:,}/{len(futs):,} ({rate:.1f}/s, ETA {(len(futs)-k)/max(rate,1e-6)/60:.1f} min)")
vis = pd.read_csv(vis_csv).merge(v[["path", "rel", "target", "source", "scene"]], on="path", how="left")
vis_err = vis[vis.error.fillna("") != ""]
log(f"Visual check done: {len(vis):,} fakes; errors: {len(vis_err)}")
if len(vis_err):
    print(vis_err.error.value_counts().head(10).to_string())

In [ ]:
# ===== CELL 11: Visual summaries per method =====
# Flags (thresholds are deliberately simple; flagged videos are shown in Cell 12 for a human look):
#   unchanged    : almost no pixels differ from the original → the "fake" may effectively be real
#   misaligned   : background differs much more than usual → fake and original frames are not the same frames
#   multi_face   : 2+ faces seen in the fake → only one is manipulated, the others are REAL faces
#   not_largest  : the manipulated face is not the biggest face in the frame
#   outside_face : most changed pixels are not inside any detected face
vis_ok = vis[vis.error.fillna("") == ""].copy()
vis_ok["unchanged"] = (vis_ok.diff_p99 < 8) | (vis_ok.changed_frac < 1e-4)
# misaligned = background differs far more than is typical for that method (compression noise sets the baseline)
vis_ok["misaligned"] = vis_ok.bg_diff > np.maximum(8, 2.5 * vis_ok.groupby("method").bg_diff.transform("median"))
vis_ok["multi_face"] = vis_ok.max_faces >= 2
vis_ok["not_largest"] = vis_ok.manip_rank >= 1
vis_ok["no_face_found"] = vis_ok.max_faces == 0
vis_ok["outside_face"] = (vis_ok.changed_in_faces < 0.5) & ~vis_ok.unchanged & (vis_ok.max_faces > 0)
FLAGS = ["unchanged", "misaligned", "multi_face", "not_largest", "no_face_found", "outside_face"]
tab = vis_ok.groupby("method").agg(checked=("path", "size"), changed_pct_med=("changed_frac", lambda x: round(100 * x.median(), 2)),
                                   bg_diff_med=("bg_diff", "median"), **{f: (f, "sum") for f in FLAGS})
print(tab.round(3).to_string())
print("\nMax faces per fake video:", vis_ok.max_faces.value_counts().sort_index().to_dict())
vis_ok.to_csv(OUT / "visual_flags.csv", index=False)
log("Saved visual_flags.csv")

In [ ]:
# ===== CELL 12: Contact sheets — look at flagged videos with your own eyes =====
# Each row: original frame | fake frame | changed pixels (red). Face boxes: green = manipulated, yellow = untouched.
def sheet(rows, title, fn, k=8):
    tiles = []
    for _, r in rows.head(k).iterrows():
        j = dict(path=r.path)
        meta_f = v.set_index("path").loc[r.path]
        okey = ("FFPP", meta_f.target) if meta_f.dataset == "FFPP" else ("DFD", f"{meta_f.target}__{meta_f.scene}")
        o = ov.loc[okey]; o = o.iloc[0] if isinstance(o, pd.DataFrame) else o
        i = [x for x in FRAME_IDX if x < min(meta_f.n_frames, o.n_frames)][:1] or [0]
        F, O = read_frames(r.path, i), read_frames(o.path, i)
        if i[0] not in F or i[0] not in O or F[i[0]].shape != O[i[0]].shape:
            continue
        f, oo = F[i[0]], O[i[0]]
        d = cv2.GaussianBlur(cv2.absdiff(f, oo).max(axis=2), (5, 5), 0)
        m = d > DIFF_THRESHOLD
        heat = oo.copy(); heat[m] = (0, 0, 255)
        fb = f.copy()
        for b in (detect_faces(f) if FACES_OK else []):
            x, y, w, h = map(int, b[:4])
            col = (0, 255, 0) if overlap_share(m.astype(np.uint8), b) > 0.3 else (0, 255, 255)
            cv2.rectangle(fb, (x, y), (x + w, y + h), col, 2)
        row = np.hstack([oo, fb, heat])
        s = 960 / row.shape[1]
        row = cv2.resize(row, (960, int(row.shape[0] * s)))
        cv2.putText(row, f"{r.method} {Path(r.path).stem}", (8, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        tiles.append(row)
    if not tiles:
        return None
    out = np.vstack(tiles)
    cv2.imwrite(str(OUT / "figures" / fn), out, [cv2.IMWRITE_JPEG_QUALITY, 85])
    log(f"  {title}: {len(tiles)} examples → figures/{fn}")
    return OUT / "figures" / fn

made = []
for flag in ["multi_face", "not_largest", "unchanged", "misaligned", "outside_face", "no_face_found"]:
    rows = vis_ok[vis_ok[flag]].sample(frac=1, random_state=SEED)
    if len(rows):
        made.append(sheet(rows, flag, f"flag_{flag}.jpg"))
fkk = fk[fk.compression == VISUAL_COMPRESSION].copy()
fkk["sp"] = fkk.speed_vs_target.sub(1).abs()
fast = vis_ok.merge(fkk[["path", "sp"]], on="path").sort_values("sp", ascending=False)
if len(fast) and fast.sp.iloc[0] > 0.01:
    made.append(sheet(fast, "biggest speed change", "flag_speed.jpg"))
for meth in sorted(vis_ok.method.unique()):          # one normal example sheet per method
    made.append(sheet(vis_ok[(vis_ok.method == meth) & ~vis_ok[FLAGS].any(axis=1)].sample(frac=1, random_state=SEED),
                      f"typical {meth}", f"typical_{meth}.jpg", k=4))
try:
    from IPython.display import Image, display
    for p in [m for m in made if m][:3]:
        display(Image(filename=str(p), width=900))
except Exception:
    pass

In [ ]:
# ===== CELL 13: Report — numbers for DECISIONS/OUTPUTS + a zip to download =====
pct = lambda a, b: f"{100*a/max(b,1):.1f}%"
def md_table(df):
    """DataFrame → Markdown table (no extra packages needed)."""
    df = df.reset_index() if not isinstance(df.index, pd.RangeIndex) else df
    cols = [str(c) for c in df.columns]
    out = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for row in df.astype(str).values.tolist():
        out.append("| " + " | ".join(row) + " |")
    return "\n".join(out)
nfk = len(fk)
lines = ["# FF++ audit report (DeepGuard v2, Phase 1.1)", "",
         f"Generated: {time.strftime('%Y-%m-%d %H:%M')} | input: {INPUT_ROOT}", "",
         "## Contents", f"- Files: {len(files):,} ({files.bytes.clip(lower=0).sum()/1e9:.1f} GB); videos: {(inv.kind=='video').sum():,}; frame folders: {(inv.kind=='frames_folder').sum():,}",
         f"- Unparsed names: {(inv.dataset=='UNPARSED').sum()}; unreadable videos: {len(errs)}; byte-identical duplicate groups: {len(dup_hash)}", "",
         "## Completeness vs official lists", md_table(completeness), "",
         f"Official-split leakage: {len(leak)} YouTube videos have sequences in different official splits: " +
         "; ".join(f"{k}: {v_}" for k, v_ in leak), "",
         "## Metadata per method", md_table(summary.round(4).reset_index()), "",
         "## H1 fps of fake vs originals", md_table(pd.crosstab([fk.method, fk.compression], fk.fps_case).reset_index()), "",
         "## H2 frame count of fake vs originals", md_table(pd.crosstab([fk.method, fk.compression], fk.frames_case).reset_index()), "",
         f"Fakes playing at a different speed than their real target: {pct((fk.speed_vs_target.sub(1).abs()>0.01).sum(), nfk)} "
         f"(range {fk.speed_vs_target.min():.3f}×–{fk.speed_vs_target.max():.3f}×)", "",
         "## Metadata-only shortcut AUC (flag if > 0.60)", md_table(shortcut), "",
         "## Visual check (c23)", md_table(tab.round(3).reset_index()), "",
         "Figures: see `figures/` (flag_*.jpg, typical_*.jpg)."]
(OUT / "FFPP_AUDIT_REPORT.md").write_text("\n".join(lines))
summary_json = dict(
    files=len(files), videos=int((inv.kind == "video").sum()), frame_folders=int((inv.kind == "frames_folder").sum()),
    unparsed=int((inv.dataset == "UNPARSED").sum()), unreadable=len(errs), duplicate_groups=len(dup_hash),
    official_split_leak=len(leak),
    fps_case=fk.groupby(["method", "compression"]).fps_case.value_counts().unstack(fill_value=0).to_dict("index"),
    frames_case=fk.groupby(["method", "compression"]).frames_case.value_counts().unstack(fill_value=0).to_dict("index"),
    shortcut=shortcut[["compression", "fakes", "auc"]].to_dict("records"),
    visual=tab.reset_index().to_dict("records"))
def jsonable(o):
    """Make nested dicts JSON-safe (tuple keys → strings)."""
    if isinstance(o, dict):
        return {str(k): jsonable(x) for k, x in o.items()}
    if isinstance(o, list):
        return [jsonable(x) for x in o]
    return o
json.dump(jsonable(summary_json), open(OUT / "audit_summary.json", "w"), indent=1, default=str)
zpath = OUT.parent / "ffpp_audit_results.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file() and p.suffix != ".onnx" and p.name != "all_files.csv":
            z.write(p, p.relative_to(OUT.parent))
log(f"Report: {OUT/'FFPP_AUDIT_REPORT.md'}")
log(f"Download this zip from the Output tab and send it to Claude: {zpath} ({zpath.stat().st_size/1e6:.1f} MB)")
print("\n" + "\n".join(lines[:12]))